# Phase 4 & 5 : Modélisation Classique et Machine Learning (ML)

Ce notebook présente la phase de modélisation classique par apprentissage automatique pour prédire les arrivées touristiques au Maroc.

## Modèles entraînés et évalués :
1. **Régression Ridge** : Linéaire régularisée L2 pour éviter le surapprentissage.
2. **SVR (Support Vector Regression)** : Modèle non linéaire robuste avec noyau RBF.
3. **XGBoost (eXtreme Gradient Boosting)** : Algorithme puissant d'arbres décisionnels boostés par gradient.
4. **LightGBM (Light Gradient Boosting Machine)** : Modèle d'arbres ultra-rapide et performant sur données tabulaires.
5. **CatBoost (Categorical Boosting)** : Performant et stable dès les hyperparamètres par défaut.

Tous les modèles sont optimisés via **GridSearchCV** combiné à un découpage temporel **TimeSeriesSplit** pour garantir la robustesse sans fuite d'informations futures.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.config import DATA_DIR, TARGET_COL, TEST_START_DATE
import src.models_ml as ml
import src.evaluation as eval_mod
import src.visualization as viz

## 1. Chargement des Données Préparées (Splits Train/Test)

In [ ]:
separated_dir = os.path.join(DATA_DIR, 'separted')

X_train = pd.read_csv(os.path.join(separated_dir, 'X_train.csv'))
y_train = pd.read_csv(os.path.join(separated_dir, 'y_train.csv')).squeeze()
X_test = pd.read_csv(os.path.join(separated_dir, 'X_test.csv'))
y_test = pd.read_csv(os.path.join(separated_dir, 'y_test.csv')).squeeze()

print(f"X_train shape: {X_train.shape} | y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape} | y_test shape: {y_test.shape}")

## 2. Entraînement et Recherche d'Hyperparamètres (GridSearchCV)

Nous lançons la recherche d'hyperparamètres sur l'ensemble des modèles de ML. Chaque modèle est encapsulé dans une `Pipeline` scikit-learn pour garantir que l'imputation des valeurs manquantes (`SimpleImputer`) et la mise à l'échelle (`StandardScaler`) sont calculées uniquement sur les plis d'entraînement internes de la validation croisée, évitant ainsi le *data leakage*.

In [ ]:
print("--- 1. Entraînement de la Régression Ridge ---")
ridge_grid = ml.train_ridge(X_train, y_train, cv=3)
print(f"Meilleurs paramètres Ridge : {ridge_grid.best_params_}")

In [ ]:
print("--- 2. Entraînement du SVR ---")
svr_grid = ml.train_svr(X_train, y_train, cv=3)
print(f"Meilleurs paramètres SVR : {svr_grid.best_params_}")

In [ ]:
print("--- 3. Entraînement d'XGBoost ---")
xgb_grid = ml.train_xgboost(X_train, y_train, cv=3)
if xgb_grid:
    print(f"Meilleurs paramètres XGBoost : {xgb_grid.best_params_}")

In [ ]:
print("--- 4. Entraînement de LightGBM ---")
lgb_grid = ml.train_lightgbm(X_train, y_train, cv=3)
if lgb_grid:
    print(f"Meilleurs paramètres LightGBM : {lgb_grid.best_params_}")

In [ ]:
print("--- 5. Entraînement de CatBoost ---")
cat_grid = ml.train_catboost(X_train, y_train, cv=3)
if cat_grid:
    print(f"Meilleurs paramètres CatBoost : {cat_grid.best_params_}")

## 3. Prédictions et Évaluation

Nous faisons des prédictions sur l'ensemble de test avec les meilleurs estimateurs identifiés et calculons les métriques de performance globales (RMSE, MAE, MAPE, R2).

In [ ]:
predictions = {}

# Récupération des prédictions de chaque modèle
predictions['Ridge'] = ridge_grid.predict(X_test)
predictions['SVR'] = svr_grid.predict(X_test)

if xgb_grid:
    predictions['XGBoost'] = xgb_grid.predict(X_test)
if lgb_grid:
    predictions['LightGBM'] = lgb_grid.predict(X_test)
if cat_grid:
    predictions['CatBoost'] = cat_grid.predict(X_test)

# Comparaison comparative
results_df = eval_mod.compare_models(predictions, y_test)
display(results_df)

## 4. Visualisation Graphique des Prévisions

Nous comparons visuellement les prédictions des modèles par rapport aux données réelles de test.

In [ ]:
# Récupération des dates associées au test set pour le tracé
df_full = pd.read_csv(os.path.join(DATA_DIR, 'merged_tourism_data_final.csv'))
df_full['Date'] = pd.to_datetime(df_full['Date'])
test_dates = df_full[df_full['Date'] >= TEST_START_DATE]['Date'].values[:len(y_test)]

viz.plot_predictions_comparison(y_test, predictions, test_dates, title="Comparaison des Modèles de ML sur le Test Set (2023-2026)")

## 5. Export des Métriques de Performance

Nous enregistrons le tableau comparatif dans un fichier CSV pour l'analyse globale finale.

In [ ]:
metrics_output_path = os.path.join(DATA_DIR, 'model_performance_metrics_ML.csv')
results_df.to_csv(metrics_output_path, index=False)
print(f"Métriques ML sauvegardées dans {metrics_output_path}")